In [1]:
import sqlite3
import pandas as pd

# Load datasets
athletes = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/athlete_events.csv")
regions = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/noc_regions.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv", sep='\t')


# Connect to SQLite in-memory DB
conn = sqlite3.connect(":memory:")

# Write DataFrames to SQL tables
athletes.to_sql("athletes_table", conn, index=False, if_exists="replace")
regions.to_sql("regions_table", conn, index=False, if_exists="replace")
sales.to_sql("sales_table", conn, index=False, if_exists="replace")

4622

In [2]:
#Visualize the athletes table
pd.read_sql("SELECT * FROM athletes_table",conn)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,None
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,None
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,None
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271111,135569,Andrzej ya,M,29.0,179.0,89.0,Poland-1,POL,1976 Winter,1976,Winter,Innsbruck,Luge,Luge Mixed (Men)'s Doubles,None
271112,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Individual",None
271113,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Team",None
271114,135571,Tomasz Ireneusz ya,M,30.0,185.0,96.0,Poland,POL,1998 Winter,1998,Winter,Nagano,Bobsleigh,Bobsleigh Men's Four,None


In [3]:
#Visualize the regions table
pd.read_sql("SELECT * FROM regions_table",conn)

,NOC,region,notes
0,AFG,Afghanistan,None
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,None
3,ALG,Algeria,None
4,AND,Andorra,None
...,...,...,...
225,YEM,Yemen,None
226,YMD,Yemen,South Yemen
227,YUG,Serbia,Yugoslavia
228,ZAM,Zambia,None


In [4]:
#Visualize the sales table
pd.read_sql("SELECT * FROM sales_table",conn)

,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,None,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,None,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98
...,...,...,...,...,...
4617,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Sour ...",$11.75
4618,1833,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Sour Cream, Cheese...",$11.75
4619,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Pinto...",$11.25
4620,1834,1,Chicken Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Lettu...",$8.75


In [5]:
#Count the total number of medals won by each country and show the top 5.
pd.read_sql(
    """
    SELECT NOC as Country , COUNT(Medal) AS No_Medals -- Select only the country and Number of Medals
    FROM athletes_table -- Refer to the athletes table
    GROUP BY NOC -- Group by the contry inorder for the count function to work as intended
    ORDER BY No_Medals DESC -- Use descending order to get top 5
    LIMIT 5 -- Only show the top 5
    """, conn )

,Country,No_Medals
0,USA,5637
1,URS,2503
2,GER,2165
3,GBR,2068
4,FRA,1777


In [6]:
#Calculate the average age of athletes who won a Gold medal.
pd.read_sql(
    """
    SELECT AVG(Age) AS Average_Age  -- Use the average function, AVG(), to get the average Age
    FROM athletes_table -- Refer to the athletes table
    WHERE Medal = 'Gold' -- Only use the athletes who won a gold medal
    """,conn
)

,Average_Age
0,25.901013


In [7]:
#How many distinct events are there in each sport?
pd.read_sql(
    """
    SELECT Sport, COUNT(DISTINCT Event) AS No_Events /* Select the Sport column and count the unique events using DISTINCT Keyword 
    ensuring that each event is counted only once if multiple athletes participated in it*/
    FROM athletes_table -- Refer to athletes table
    GROUP BY Sport -- Allows us to get count per sport
    """,conn
)

,Sport,No_Events
0,Aeronautics,1
1,Alpine Skiing,10
2,Alpinism,1
3,Archery,29
4,Art Competitions,29
...,...,...
61,Tug-Of-War,1
62,Volleyball,2
63,Water Polo,2
64,Weightlifting,21


In [8]:
#Show all athletes from the United States (NOC = 'USA')
pd.read_sql(
    """
    SELECT * -- Show everything about the athlete
    FROM athletes_table -- Refer to the athletes table
    WHERE NOC = 'USA' -- Only show the athltes from the United States
    """,conn
)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,6,Per Knut Aaland,M,31.0,188.0,75.0,United States,USA,1992 Winter,1992,Winter,Albertville,Cross Country Skiing,Cross Country Skiing Men's 10 kilometres,None
1,6,Per Knut Aaland,M,31.0,188.0,75.0,United States,USA,1992 Winter,1992,Winter,Albertville,Cross Country Skiing,Cross Country Skiing Men's 50 kilometres,None
2,6,Per Knut Aaland,M,31.0,188.0,75.0,United States,USA,1992 Winter,1992,Winter,Albertville,Cross Country Skiing,Cross Country Skiing Men's 10/15 kilometres Pu...,None
3,6,Per Knut Aaland,M,31.0,188.0,75.0,United States,USA,1992 Winter,1992,Winter,Albertville,Cross Country Skiing,Cross Country Skiing Men's 4 x 10 kilometres R...,None
4,6,Per Knut Aaland,M,33.0,188.0,75.0,United States,USA,1994 Winter,1994,Winter,Lillehammer,Cross Country Skiing,Cross Country Skiing Men's 10 kilometres,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18848,135458,Rami Zur,M,27.0,175.0,77.0,United States,USA,2004 Summer,2004,Summer,Athina,Canoeing,"Canoeing Men's Kayak Doubles, 500 metres",None
18849,135458,Rami Zur,M,31.0,175.0,77.0,United States,USA,2008 Summer,2008,Summer,Beijing,Canoeing,"Canoeing Men's Kayak Singles, 500 metres",None
18850,135458,Rami Zur,M,31.0,175.0,77.0,United States,USA,2008 Summer,2008,Summer,Beijing,Canoeing,"Canoeing Men's Kayak Singles, 1,000 metres",None
18851,135543,"Victor Andrew ""Vic"" Zwolak",M,25.0,175.0,64.0,United States,USA,1964 Summer,1964,Summer,Tokyo,Athletics,"Athletics Men's 3,000 metres Steeplechase",None


In [9]:
#Count how many medals were awarded each year
pd.read_sql(
    """
    SELECT Year,COUNT(Medal) AS No_Medals  -- Select the Year column and counts the Medal Coulumn entries
    FROM athletes_table -- Refer to the athletes table
    GROUP BY Year -- Allows us to get count per year
    """,conn
)

,Year,No_Medals
0,1896,143
1,1900,604
2,1904,486
3,1906,458
4,1908,831
5,1912,941
6,1920,1308
7,1924,962
8,1928,823
9,1932,739


In [10]:
#Find all athlete records where height or weight is missing
pd.read_sql(
    """
    SELECT * -- Select all columns
    FROM athletes_table -- Refer to the athletes table
    WHERE Height IS NULL OR Weight IS NULL -- Find records where either Height or Weight is missing
    """,conn
)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,None
1,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
2,8,"Cornelia ""Cor"" Aalten (-Strannood)",F,18.0,168.0,NaN,Netherlands,NED,1932 Summer,1932,Summer,Los Angeles,Athletics,Athletics Women's 100 metres,None
3,8,"Cornelia ""Cor"" Aalten (-Strannood)",F,18.0,168.0,NaN,Netherlands,NED,1932 Summer,1932,Summer,Los Angeles,Athletics,Athletics Women's 4 x 100 metres Relay,None
4,10,"Einar Ferdinand ""Einari"" Aalto",M,26.0,NaN,NaN,Finland,FIN,1952 Summer,1952,Summer,Helsinki,Swimming,Swimming Men's 400 metres Freestyle,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64258,135539,Marius Edmund Zwiller,M,18.0,NaN,NaN,France,FRA,1924 Summer,1924,Summer,Paris,Swimming,Swimming Men's 200 metres Breaststroke,None
64259,135542,Werner Zwingli,M,29.0,NaN,NaN,Switzerland,SUI,1956 Winter,1956,Winter,Cortina d'Ampezzo,Cross Country Skiing,Cross Country Skiing Men's 15 kilometres,None
64260,135542,Werner Zwingli,M,29.0,NaN,NaN,Switzerland,SUI,1956 Winter,1956,Winter,Cortina d'Ampezzo,Cross Country Skiing,Cross Country Skiing Men's 4 x 10 kilometres R...,None
64261,135552,Jan (Johann-) Zybert (Siebert-),M,20.0,NaN,NaN,Poland,POL,1928 Summer,1928,Summer,Amsterdam,Cycling,"Cycling Men's Team Pursuit, 4,000 metres",None


In [11]:
#Replace the missing height with the average athlete height
pd.read_sql(
    """
    SELECT 
        ID, Name, Sex, Age, -- Select columns from the athlete's table
        ROUND(COALESCE(Height, (SELECT AVG(Height) FROM athletes_table)),1) AS Height, -- Use the coalesce function to fill in the missing values for those columns which have a missing value for height
        Weight, Team, NOC, Games, Year, Season, City, Sport, Event, Medal -- Select remaining columns
    FROM athletes_table -- Refer to the athletes table
    """,conn
)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,None
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,None
2,3,Gunnar Nielsen Aaby,M,24.0,175.3,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,None
3,4,Edgar Lindenau Aabye,M,34.0,175.3,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271111,135569,Andrzej ya,M,29.0,179.0,89.0,Poland-1,POL,1976 Winter,1976,Winter,Innsbruck,Luge,Luge Mixed (Men)'s Doubles,None
271112,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Individual",None
271113,135570,Piotr ya,M,27.0,176.0,59.0,Poland,POL,2014 Winter,2014,Winter,Sochi,Ski Jumping,"Ski Jumping Men's Large Hill, Team",None
271114,135571,Tomasz Ireneusz ya,M,30.0,185.0,96.0,Poland,POL,1998 Winter,1998,Winter,Nagano,Bobsleigh,Bobsleigh Men's Four,None


In [12]:
#Return total sales value per item
pd.read_sql(
    """
    SELECT 
        item_name, -- Select item_name
        SUM(CAST(REPLACE(item_price, '$', '') AS FLOAT)) AS total_sales_value -- use replace to remove the '$', then use cast to convert the resulting string to a float then sum 
    FROM sales_table -- Refer to sales table
    GROUP BY item_name -- Allows us to do total sales value for each unique item
    ORDER BY total_sales_value DESC -- Order the total sales colun in descending order
    """,conn
)

,item_name,total_sales_value
0,Chicken Bowl,7342.73
1,Chicken Burrito,5575.82
2,Steak Burrito,3851.43
3,Steak Bowl,2260.19
4,Chips and Guacamole,2201.04
5,Chicken Salad Bowl,1228.75
6,Chicken Soft Tacos,1108.09
7,Veggie Burrito,934.77
8,Barbacoa Burrito,894.75
9,Veggie Bowl,867.99


In [13]:
#Show the top 5 records with the highest item_price
pd.read_sql(
    """
    SELECT * -- Select everything from the table
    FROM sales_table -- Refer to sales table
    ORDER BY CAST(REPLACE(item_price, '$', '') AS FLOAT) DESC -- Remove the '$' and convert to float to enable us to use the Descending option effectively
    LIMIT 5 -- Only Show 5 records
    """,conn
)

,order_id,quantity,item_name,choice_description,item_price
0,1443,15,Chips and Fresh Tomato Salsa,None,$44.25
1,1398,3,Carnitas Bowl,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",$35.25
2,511,4,Chicken Burrito,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",$35.00
3,1443,4,Chicken Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Chees...",$35.00
4,1443,3,Veggie Burrito,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",$33.75


In [14]:
#How many unique customer orders are there
pd.read_sql(
    """
    SELECT COUNT(DISTINCT order_id) AS unique_orders -- Use distinct to only count unique order ids
    FROM sales_table -- Refers to sales table
    """,conn
)

,unique_orders
0,1834


In [15]:
pd.read_sql(
    """
    WITH medalists AS (
        -- Find all unique athletes who won at least one medal
        SELECT DISTINCT ID, Name, Age, NOC
        FROM athletes_table
        WHERE Medal IS NOT NULL
    )
    -- Main query
    SELECT 
        r.region AS Country,
        COUNT(DISTINCT m.ID) AS Num_Medalists, -- Count the number of athletes who won at least one medal
        ROUND(AVG(m.Age), 2) AS Avg_Age_Medalists, -- Calculate average age of medalists
        CASE 
            WHEN AVG(m.Age) < 25 THEN 'High'
            WHEN AVG(m.Age) BETWEEN 25 AND 30 THEN 'Medium'
            ELSE 'Low'
        END AS Performance -- Categorize performance based on average age
    FROM medalists m
    JOIN regions_table r ON m.NOC = r.NOC -- JOIN athletes with regions to get country names
    GROUP BY r.region -- Group by country to aggregate statistics
    ORDER BY Avg_Age_Medalists ASC -- Order by average age for easier analysis
    """,conn
)

,Country,Num_Medalists,Avg_Age_Medalists,Performance
0,Iraq,1,NaN,Low
1,Nepal,1,NaN,Low
2,Botswana,1,18.00,High
3,Curacao,1,19.00,High
4,Ghana,23,19.17,High
...,...,...,...,...
131,Djibouti,1,31.00,Low
132,Saudi Arabia,6,32.00,Low
133,United Arab Emirates,2,34.50,Low
134,Individual Olympic Athletes,5,34.60,Low
